In [ ]:
# QC, scrublet and CellTypist benchmark on an adipose reference dataset.
import scanpy as sc
import celltypist as ct
from celltypist import models

In [ ]:
import scanpy as sc
import celltypist as ct
from celltypist import models
import anndata as ad

In [ ]:
model_loaded = ct.models.Model.load("/home/yanjun3/owen/GSE155960/WAT_custom.pkl")

In [ ]:
model_loaded.cell_types

In [ ]:
import os
input_dir = "/home/yanjun3/atlas/adipose_sc"
model = model_loaded

for fname in os.listdir(input_dir):
    if fname.endswith(".h5ad") and not fname.startswith(("annotated_", "subsetted_")):
        file_path = os.path.join(input_dir, fname)
        adata = sc.read(file_path)

        # Quality Control
        adata.var["mt"] = adata.var_names.str.upper().str.startswith("MT-")
        sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)
        sc.pp.filter_cells(adata, min_genes=300)
        adata = adata[adata.obs["pct_counts_mt"] < 20].copy()
        sc.pp.filter_genes(adata, min_cells=10)
        sc.pp.scrublet(adata)

        # Normalization
        adata.layers["counts"] = adata.X.copy()
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        adata.layers["log_counts"] = adata.X.copy()

        # CellTypist annotation (labels only)
        result = ct.annotate(adata, model=model, majority_voting=True)
        adata.obs["celltypist_labels"] = result.predicted_labels["majority_voting"]

        # Save annotated file
        out_path = os.path.join(input_dir, f"annotated_{fname}")
        adata.write(out_path)



In [ ]:
import pandas as pd
input_dir = "/home/yanjun3/atlas/adipose_sc"

# collect annotated files
files = [f for f in os.listdir(input_dir) if f.startswith("annotated_") and f.endswith(".h5ad")]

adatas, keys = [], []
for fname in files:
    ad = sc.read_h5ad(os.path.join(input_dir, fname))
    if ad.n_obs == 0:
        continue
    sample = fname.replace("annotated_", "").replace(".h5ad", "")
    ad.obs["sample"] = sample
    adatas.append(ad)
    keys.append(sample)

# concatenate
adipose_sc_adata = sc.concat(adatas, label="sample", keys=keys, index_unique="-", join="outer")

# join external metadata (do not replace obs)
info = pd.read_csv("/home/yanjun3/atlas/adipose_sc/adipose_sc_clean.csv").drop_duplicates(subset=["sample_id"])
adipose_sc_adata.obs = adipose_sc_adata.obs.join(info.set_index("sample_id"), on="sample")
adipose_sc_adata.write(os.path.join(input_dir, "adipose_sc_all.h5ad"))

In [ ]:
import scanpy as sc
import celltypist as ct
from celltypist import models
import anndata as ad

In [ ]:
adata = sc.read('/home/yanjun3/atlas/adipose_sc/adipose_sc_all.h5ad')

In [ ]:
adata.obs_names_make_unique()

In [ ]:
import scvi
import seaborn as sns
import torch
from rich import print
from scib_metrics.benchmark import Benchmarker

#unintegrated

In [ ]:
sc.set_figure_params(figsize=(6, 6), frameon=False, dpi = 300)
sc.settings.figdir = "/home/yanjun3/atlas/intestine"
sns.set_theme()
torch.set_float32_matmul_precision("high")

%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"

In [ ]:
sc.pp.highly_variable_genes(adata)
sc.tl.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
adata

In [ ]:
label_key = "celltypist_labels"
batch_key = "sample"

In [ ]:
sc.pl.umap(adata, color = [batch_key])

In [ ]:
adata_hvg = adata[:, adata.var["highly_variable"]].copy()
adata_hvg

In [ ]:
adata_scvi = adata_hvg.copy()

In [ ]:
scvi.model.SCVI.setup_anndata(adata_scvi, layer="counts", batch_key=batch_key)
adata_scvi

In [ ]:
model_scvi = scvi.model.SCVI(adata_scvi)
model_scvi

In [ ]:
scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)

In [ ]:
model_scvi.view_anndata_setup()

In [ ]:
import numpy as np
max_epochs_scvi = np.min([round((20000 / adata.n_obs) * 400), 400])
max_epochs_scvi

In [ ]:
model_scvi.train()

In [ ]:
adata_scvi.obsm["X_scVI"] = model_scvi.get_latent_representation()

In [ ]:
sc.pp.neighbors(adata_scvi, use_rep="X_scVI")
sc.tl.umap(adata_scvi)
adata_scvi

In [ ]:
sc.pl.umap(adata_hvg, color=[label_key, batch_key], wspace=1)

In [ ]:
sc.pl.umap(adata_scvi, color=[label_key, batch_key], wspace=1)

##scanvi

In [ ]:
model_scanvi = scvi.model.SCANVI.from_scvi_model(
    model_scvi, labels_key='celltypist_labels', unlabeled_category="unlabelled"
)
print(model_scanvi)
model_scanvi.view_anndata_setup()

In [ ]:
import numpy as np

In [ ]:
max_epochs_scanvi = int(np.min([10, np.max([2, round(max_epochs_scvi / 3.0)])]))
print(max_epochs_scanvi)

In [ ]:
model_scanvi.train(max_epochs=max_epochs_scanvi)

In [ ]:
sc.pl.umap(
    adata_scanvi,
    color=[label_key, batch_key],
    ncols=1,           # <-- fix
      # horizontal space between subplots
)


In [ ]:
adata_scanvi = adata_scvi.copy()
adata_scanvi.obsm["X_scANVI"] = model_scanvi.get_latent_representation()
sc.pp.neighbors(adata_scanvi, use_rep="X_scANVI")
sc.tl.umap(adata_scanvi)
sc.pl.umap(adata_scanvi, color=[label_key, batch_key], wspace=1)

####bbknn

In [ ]:
neighbors_within_batch = 25 if adata_hvg.n_obs > 100000 else 3
neighbors_within_batch

In [ ]:
adata_bbknn = adata_hvg.copy()
adata_bbknn.X = adata_bbknn.layers["log_counts"].copy()
sc.pp.pca(adata_bbknn)

In [ ]:
import bbknn
bbknn.bbknn(
    adata_bbknn, batch_key=batch_key, neighbors_within_batch=neighbors_within_batch
)
adata_bbknn

In [ ]:
sc.tl.umap(adata_bbknn)
sc.pl.umap(adata_bbknn, color=[label_key, batch_key], wspace=1)

In [ ]:
import scib
metrics_scvi = scib.metrics.metrics_fast(
    adata, adata_scvi, batch_key, label_key, embed="X_scVI"
)
metrics_scanvi = scib.metrics.metrics_fast(
    adata, adata_scanvi, batch_key, label_key, embed="X_scANVI"
)
metrics_bbknn = scib.metrics.metrics_fast(adata, adata_bbknn, batch_key, label_key)
metrics_hvg = scib.metrics.metrics_fast(adata, adata_hvg, batch_key, label_key)

In [ ]:
import pandas as pd
metrics = pd.concat(
    [metrics_scvi, metrics_scanvi, metrics_bbknn, metrics_hvg],
    axis="columns",
)
# Set methods as column names
metrics = metrics.set_axis(
    ["scVI", "scANVI", "BBKNN", "Unintegrated"], axis="columns"
)
# Select only the fast metrics
metrics = metrics.loc[
    [
        "ASW_label",
        "ASW_label/batch",
        "PCR_batch",
        "isolated_label_silhouette",
        "graph_conn",
        "hvg_overlap",
    ],
    :,
]
# Transpose so that metrics are columns and methods are rows
metrics = metrics.T
# Remove the HVG overlap metric because it's not relevant to embedding outputs
metrics = metrics.drop(columns=["hvg_overlap"])
metrics

In [ ]:
metrics.style.background_gradient(cmap="Blues")

In [ ]:
metrics_scaled = (metrics - metrics.min()) / (metrics.max() - metrics.min())
metrics_scaled.style.background_gradient(cmap="Blues")

In [ ]:
metrics_scaled["Batch"] = metrics_scaled[
    ["ASW_label/batch", "PCR_batch", "graph_conn"]
].mean(axis=1)
metrics_scaled["Bio"] = metrics_scaled[["ASW_label", "isolated_label_silhouette"]].mean(
    axis=1
)
metrics_scaled.style.background_gradient(cmap="Blues")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
metrics_scaled.plot.scatter(
    x="Batch",
    y="Bio",
    c=range(len(metrics_scaled)),
    ax=ax,
)

for k, v in metrics_scaled[["Batch", "Bio"]].iterrows():
    ax.annotate(
        k,
        v,
        xytext=(6, -3),
        textcoords="offset points",
        family="sans-serif",
        fontsize=12,
    )

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score

scores = {
    "scVI": {
        "Batch (sample)": silhouette_score(adata.obsm["X_scVI"], adata.obs["sample"].astype(str)),
        "Cell Type": silhouette_score(adata.obsm["X_scVI"], adata.obs["celltypist_labels"].astype(str))
    },
    "scANVI": {
        "Batch (sample)": silhouette_score(adata.obsm["X_scANVI"], adata.obs["sample"].astype(str)),
        "Cell Type": silhouette_score(adata.obsm["X_scANVI"], adata.obs["celltypist_labels"].astype(str))
    }
}

# Create DataFrame
df_scores = pd.DataFrame(scores).T

df_scores.plot(kind="bar", figsize=(8, 5), rot=0)
plt.title("Silhouette Scores: scVI vs scANVI")
plt.ylabel("Silhouette Score")
plt.axhline(0, linestyle="--", color="gray")
plt.tight_layout()
plt.show()


In [ ]:
adata.write('/home/yanjun3/atlas/intestine/trained_final.h5ad')